# Folder 01 / file 10 — smoke Adult @dev (after Compare; not after promotion_ready)

UCI Adult Census Income ([dataset](https://archive.ics.uci.edu/dataset/2/adult)): binary target `income_gt_50k` where **`>50K` = 1** and **`<=50K` = 0**. Fourteen census features; official split is `adult.data` (train) / `adult.test` (holdout).

Inlined replica of `src/n01_dev_train/n10_smoke.py`. Run cells **in order** (local or Jobs). Catalog/schema/model/mode come from task env.

Smoke-score the Adult @dev model on Feature Store keys.


## 1 — Imports


In [ ]:
from src.n00_shared.feature_store import score_keys
from src.n00_shared.runtime import Settings, assert_dev_ml_allowed, configure_mlflow, load_settings, mlflow_client
from src.n00_shared.serving import serving_config, upsert_endpoint


## 2 — `settings = load_settings()`


In [ ]:
settings = load_settings()


## 3 — `run()` step 1/2


In [ ]:
assert_dev_ml_allowed(settings)
configure_mlflow(settings)
client = mlflow_client()
version = str(client.get_model_version_by_alias(settings.source_model_name, "dev").version)
uri = f"models:/{settings.source_model_name}@dev"
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
if not spark.catalog.tableExists(settings.feature_fq):
    raise RuntimeError(f"feature store table missing: {settings.feature_fq}")
keys = spark.table(settings.feature_fq).select(settings.id_col).limit(20)
pred = score_keys(uri, keys, result_type="double")
if pred.count() != keys.count():
    raise RuntimeError("smoke prediction length mismatch")


## 4 — `run()` step 2/2


In [ ]:
try:
    upsert_endpoint(
        settings.endpoint_name,
        serving_config(settings.source_model_name, version, previous_version=None, canary_percent=100),
    )
    print(f"smoke pinned dev endpoint {settings.endpoint_name} to v{version}")
except Exception as exc:
    print(f"smoke feature-store path ok; endpoint pin skipped: {exc}")
